# DragonScope GPU training

1. In Colab, choose **Runtime > Change runtime type > T4 GPU**.
2. Choose a run in the first code cell, then run all cells.
   - `main`: the demo model, on conversations of up to 512 bytes. About 5 hours.
   - `long`: conversations padded with small talk to up to 2,048 bytes, so recall can be tested across longer sessions. Expect it to take longer. If the GPU runs out of memory, lower `BATCH`.
3. If the session dies, run all again. Training resumes from the checkpoint in Google Drive.
4. Download `best_model.pt` from `MyDrive/dragonscope-runs/gpu-<run>/`. Commit the main run as `models/best_model.pt` and the long run as `models/long_model.pt`.

In [ ]:
RUN = 'main'  # or 'long'

import os, shutil
from google.colab import drive

SETTINGS = {
    'main': {'data': '', 'batch': 32},
    'long': {'data': '--max-bytes 2048 --pad-to 2048', 'batch': 8},
}
DATA_ARGS = SETTINGS[RUN]['data']
BATCH = SETTINGS[RUN]['batch']

drive.mount('/content/drive')
RUN_DIR = f'/content/drive/MyDrive/dragonscope-runs/gpu-{RUN}'
os.makedirs(RUN_DIR, exist_ok=True)

shutil.rmtree('/content/dragonscope', ignore_errors=True)
!git clone -q https://github.com/theoreticalraccoon/dragonscope /content/dragonscope
os.chdir('/content/dragonscope')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m dragonscope.data --out data {DATA_ARGS}
!python -m pip install -q pytest && python -m pytest -q tests/test_engine.py

In [ ]:
!python -m dragonscope.train --data data --out "{RUN_DIR}" --iters 30000 --batch {BATCH} --eval-every 500 --save-every-seconds 300